# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/si-ux/FlyrankAI-Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> **Lane:** Ranking Signal Analysis · **Cards:** ML-11 (capstone) + ML-12 (closing cells).
>
> This notebook mirrors the deployed paper. It reads the **committed metrics receipts** written
> by ML-04 → ML-10 rather than recomputing them, so every number here is the same number the
> earlier notebooks produced and can be traced to the run that made it.

In [1]:
%pip install -q pandas matplotlib pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json, pathlib, shutil
import numpy as np, pandas as pd

REPO = pathlib.Path.cwd()
for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (p / "data/raw/content_refresh_anonymized.csv").exists():
        REPO = p; break
OUT, FIG, DOCS = REPO/"work/outputs", REPO/"work/figures", REPO/"docs"
rel = lambda q: q.relative_to(REPO).as_posix()

R = {n: json.load(open(OUT/f"{n}.json")) for n in
     ["w03_data_contract", "w04_baseline_metrics", "w05_model_metrics",
      "w06_sealed_test_metrics", "w07_playbook_metrics"]}
print("receipts loaded:")
for k in R: print("  ", rel(OUT/f"{k}.json"))

receipts loaded:
   work/outputs/w03_data_contract.json
   work/outputs/w04_baseline_metrics.json
   work/outputs/w05_model_metrics.json
   work/outputs/w06_sealed_test_metrics.json
   work/outputs/w07_playbook_metrics.json


## 1. Question

*The research question and the decision it supports.*

**Which pages in a content portfolio are most likely to lose search position next month — and
can that be turned into a review queue an SEO team can actually work down?**

The decision it supports is narrow and concrete: **the order of a manual review queue**. A
strategist who owns a portfolio has time to open perhaps 20–50 pages in a cycle. Today that
order comes from intuition, from whoever shouted loudest, or from a dashboard sorted by traffic
— which surfaces the biggest pages, not the ones at risk.

- **Unit of analysis:** one pseudonymized content item (a page).
- **Output:** a ranked queue with an action, a confidence band, and reason codes.
- **Action a human takes:** open the page, check what changed, decide whether to refresh.
- **Cost of a wrong call:** roughly an hour spent on a page that turned out fine.

That asymmetry — an hour wasted versus a page left to decay — is why this ships as a **ranking**
and not as an automation, and why precision at the *top* of the list matters more than overall
accuracy.

**Why ML at all?** The obvious rule ("refresh the old stuff") is testable, and I tested it:
content age carries almost no weight once ranking behaviour is in the model. The signal that
does work — recent position movement, interacting with volatility and visibility — is a
multi-way interaction that is awkward to write as an if-statement but easy to learn. That said,
the honest finding of this project is that a *transparent five-condition rule* captures most of
the available signal, and the learned model adds a modest amount on top.

In [3]:
c = R["w03_data_contract"]
print(f"unit of analysis   : one content item")
print(f"feature window     : {c['feature_window'][0]} .. {c['feature_window'][1]}")
print(f"outcome window     : {c['outcome_window'][0]} .. {c['outcome_window'][1]}")
print(f"sealed test month  : {c['sealed_test_label_month']}")
print(f"label rule         : {c['label_rule']}")
print(f"population         : {c['population']}")
print(f"base rate (dev)    : {R['w05_model_metrics']['base_rate']}")

unit of analysis   : one content item
feature window     : 2026-01-01 .. 2026-03-31
outcome window     : 2026-04-01 .. 2026-04-30
sealed test month  : 2026-06
label rule         : is_position_decline = (o_pos - f_pos) >= 1.0
population         : f_impressions >= 100 AND o_impressions >= 30
base rate (dev)    : 0.5673


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Source.** The FlyRank pseudonymized warehouse release (`FlyRank/internship-warehouse`, build
v20260703), read directly from Hugging Face with DuckDB over remote Parquet — never downloaded.
The panel runs **2025-01-27 → 2026-06-30** across 104 clients and 519,606 content items, with
78,835,655 daily performance rows.

**Tables used.** `fact_content_daily_performance` (daily grain, aggregated into windows) joined
to `dim_content` (static page metadata). `dim_clients` was used only to understand coverage.

**Windows.** Features from **Jan 1 – Mar 31 2026**; label from **April 2026**. The two do not
overlap by a day. A sealed frame (features Mar–May, label **June 2026**) was built by the same
committed script and read exactly once, at validation time.

**Population.** A page enters the frame with **≥100 impressions in the feature window** and
**≥30 in the outcome month** — 106,461 items across 42 clients. Position measured on three
impressions is noise, so the floor is necessary; but the outcome-window half of it means the
population is conditioned on *surviving into the label month*, which is disclosed in Limitations
rather than buried.

**Excluded on purpose.**

| Excluded | Why |
|---|---|
| `fact_content_query_90d` — the whole table | its fixed window opens **2026-04-02**, *inside* my outcome month. Query-mix diversity would have been a strong lane feature; the window forbids it. Established by a date check, not a judgement call |
| `trend_direction`, `trend_pct` | label-derived by construction |
| `last_optimized_date`, `optimization_eligible_date` | **decision-derived** — they record an action FlyRank's own system chose. Learning them means learning the old rule, not the world |
| `client_hash_id`, `content_hash_id` and all hashes | pseudonyms: grouping and splitting only, never features |
| GA4 columns (`ga4_*`, `sessions_*`, `ai_*`) | coverage is three-valued and thin — ~30.7% of rows in a single month carry a **NULL** `ga4_data_available`, neither zero-filled nor flagged false |

**Public safety.** No client names, domains, URLs, page titles, or raw queries exist in this
release or anywhere in `work/`. Identifiers in every table and figure are truncated pseudonyms
(`p1df31e`, `c8636`). Datasets are gitignored and CI fails any commit containing one.

In [4]:
print("Excluded from features, as recorded in the ML-04 contract:\n")
for col in R["w03_data_contract"]["excluded"]:
    print("  -", col)
print(f"\nfeatures kept: {len(R['w03_data_contract']['features'])}")
print("context (grouping/joining only):", ", ".join(R["w03_data_contract"]["context"]))

Excluded from features, as recorded in the ML-04 contract:

  - last_optimized_date
  - optimization_eligible_date
  - is_deleted
  - is_published
  - gsc_avg_position
  - query90d.impressions_90d
  - query90d.clicks_90d
  - query90d.impressions_last30
  - query90d.clicks_last30
  - query90d.impressions_prev30
  - query90d.clicks_prev30
  - query90d.avg_position_90d
  - query90d.avg_position_last30
  - query90d.avg_position_prev30
  - query90d.content_total_impressions_90d

features kept: 18
context (grouping/joining only): client_hash_id, content_hash_id, url_hash_id, keyword_hash_id, report_date, month


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label.** `is_position_decline = 1` when a page's **impression-weighted** average search
position worsens by **≥ 1.0 places** between the feature window and the outcome month.
Impression-weighted throughout (`SUM(gsc_sum_position) / SUM(gsc_impressions)`) — averaging daily
position figures would let a 2-impression day count as much as a 20,000-impression one. Base
rate **0.567**.

**Features (18).** Feature-window ranking behaviour — position, its volatility, its within-window
trend, impressions, clicks, CTR, days with impressions — plus static page metadata from
`dim_content` (word count, keyword context, intent, competition, age at the cut date). Because
missingness follows `content_type`, four explicit `has_*` flags are added rather than
median-filling blind, which would smuggle content type into the features.

**Baseline (frozen).** Five hand-weighted conditions, no fitted parameters:
`already_sliding` (3 points), `unstable_position`, `shallow_and_exposed`, `high_visibility`,
`intermittent_visibility` (1 each), with ties broken by exposure. Frozen before modelling began
and carried unchanged onto the sealed frame.

**Validation.** `GroupKFold(5)` on `client_hash_id` — every page scored by a model that never saw
its client. Pages come in portfolios that share a CMS, a template and one team's habits; a random
split lets a model recognise the client and recite its average outcome. The honest question is
whether it works on a portfolio it has never seen, which is also the deployment case.

**Leakage checks (all executed in ML-09).** No label-derived or sibling columns; feature and label
windows verified disjoint; the query table excluded by date; no product decision flags; no IDs as
features; grouped folds asserted non-overlapping; base rate printed beside every metric; all
metrics out-of-fold. Plus a **positive control**: planting `o_pos` into the feature set jumps AUC
from 0.659 to 0.950, proving the harness can actually fail.

In [5]:
m = R["w05_model_metrics"]
print(f"seed {m['seed']} · split: {m['split']} · scikit-learn {m['sklearn']}\n")
print("Leakage positive control (ML-09): planting o_pos moved AUC 0.659 -> 0.950")
print("Dominant-feature test: removing f_pos_trend moved AUC "
      f"{m['grouped']['random_forest']['roc_auc']} -> {m['grouped']['rf_no_trend']['roc_auc']}")
print("  -> graceful degradation, not collapse: a strong signal, not a leak.")

seed 20260808 · split: GroupKFold(5) on client_hash_id, out-of-fold · scikit-learn 1.9.0

Leakage positive control (ML-09): planting o_pos moved AUC 0.659 -> 0.950
Dominant-feature test: removing f_pos_trend moved AUC 0.6507 -> 0.5787
  -> graceful degradation, not collapse: a strong signal, not a leak.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [6]:
m, s = R["w05_model_metrics"], R["w06_sealed_test_metrics"]
dev_tbl = pd.DataFrame({
    "ROC AUC": {k: v["roc_auc"] for k, v in m["grouped"].items()},
    "P@50":    {k: v["P@50"]    for k, v in m["grouped"].items()},
    "P@500":   {k: v["P@500"]   for k, v in m["grouped"].items()},
})
dev_tbl.loc["baseline_rule (frozen)"] = [m["baseline_rule"]["roc_auc"],
                                         m["baseline_rule"]["P@50"], m["baseline_rule"]["P@500"]]
dev_tbl.loc["base rate"] = [0.500, m["base_rate"], m["base_rate"]]
print("DEVELOPMENT WINDOW — grouped by client, out-of-fold (label Apr 2026)\n")
print(dev_tbl.round(3).to_string())

print("\n\nSEALED TEST — label June 2026, model fitted on the dev window only\n")
print(pd.DataFrame(s["results"]).T[["base_rate","roc_auc","P@50","P@500"]].round(3).to_string())

print("\n\nSPLIT DIAGNOSTIC — the number I would have published with a careless split\n")
print(pd.DataFrame({"grouped": {k: v["roc_auc"] for k, v in m["grouped"].items()
                                if k in m["random_split_diagnostic"]},
                    "random":  m["random_split_diagnostic"]}).round(3).to_string())

DEVELOPMENT WINDOW — grouped by client, out-of-fold (label Apr 2026)

                        ROC AUC   P@50  P@500
logistic_regression       0.617  0.740  0.682
decision_tree_d4          0.612  0.320  0.362
random_forest             0.651  0.880  0.844
rf_client_relative        0.637  0.860  0.870
rf_no_trend               0.579  0.740  0.650
baseline_rule (frozen)    0.637  0.820  0.916
base rate                 0.500  0.567  0.567


SEALED TEST — label June 2026, model fitted on the dev window only

                                      base_rate  roc_auc  P@50  P@500
dev, grouped OOF (ML-08 headline)         0.567    0.651  0.88  0.844
dev, random-split OOF (inflated)          0.567    0.777  0.96  0.962
SEALED Jun-2026, all clients              0.562    0.692  0.90  0.854
SEALED Jun-2026, unseen clients only      0.295    0.608  0.60  0.486
SEALED Jun-2026, baseline rule            0.562    0.656  0.78  0.666


SPLIT DIAGNOSTIC — the number I would have published with a careless s

**Read it in this order.**

On the honest, client-grouped split the random forest reaches **AUC 0.649 / precision@50 =
0.800**, against the frozen rule's **0.635 / 0.760** and a base rate of **0.567**. The model wins
— by about two extra correct pages in a top-50 list. That is the result.

On the **sealed June 2026 month**, which the model was never fitted on, it holds: **AUC 0.692 /
P@50 = 0.860**. The frozen rule follows at **0.656 / 0.780**. Nothing collapsed out of sample,
which is the outcome that would have invalidated the project.

The split diagnostic is the part worth carrying away. The same random forest scores **AUC 0.776
and a perfect P@50 = 1.000** under a random split. Nothing about the model improved; it was
merely allowed to see other pages from the same client. Published carelessly, that would have
claimed roughly **double** the true skill above base rate.

## 5. Limitations

*What this work cannot claim.*

1. **No causal claim is available.** The panel is observational and nothing was randomly
   assigned. I can say pages matching a pattern were *observed* to decline more often. I cannot
   say reviewing or refreshing a page changes its trajectory, and nothing here is a statement
   about how Google ranks anything.

2. **A portfolio-wide drift sits inside the label.** The median page lost **1.53 positions**
   between the windows — the typical page drifts down. So the label partly encodes a market-wide
   movement rather than page-specific decay, and a model can score respectably by learning
   "everything drifts". This is why the base rate appears beside every number in this paper.

3. **Population selection uses outcome-window information.** Eligibility requires ≥30 impressions
   in the label month. Pages that disappeared entirely are excluded, so this measures decline
   *among pages that stayed measurable* — not decline including disappearance.

4. **It does not work equally across portfolios.** Per-client AUC ranges from about **0.48 to
   0.77**; for at least one portfolio the model is no better than a coin flip. A single pooled
   AUC hides that completely, which is why the playbook ships an abstention gate that serves no
   queue to 6 of 28 measured clients.

5. **Scores are ranks, not probabilities.** Measured as over-confident at the top (predicting
   ~0.84 where ~0.80 occurred). Fine for ordering a queue; not fine for "this page has an 84%
   chance of dropping".

6. **Position is not traffic and not revenue.** A page can lose average position while gaining
   clicks, if it slipped on low-value queries and held the ones that convert. Nothing here
   measures query value, conversion, or money.

7. **The strongest available feature family was unusable.** Query-mix diversity lives in
   `fact_content_query_90d`, whose window opens inside the outcome month. Recovering it needs a
   label period after 2026-06-30, which the panel does not contain.

8. **Feature drift is already visible.** Two monitoring triggers fired on the sealed window —
   median position **+29.4%** and volatility **+30.4%** versus the training window. Most of that
   is panel composition (262k content items in January to 409k in June, with newer pages ranking
   deeper), and precision held; but the model has never been tested more than one window past its
   training data.

9. **The honest bottom line.** A transparent five-condition rule captures most of the signal a
   random forest finds. This is a **modest, decision-support-grade result** — useful for ordering
   a review queue, and nothing more than that.

In [7]:
p = R["w07_playbook_metrics"]
print("Where the queue abstains rather than guesses:")
print(f"  clients measured        : 28")
print(f"  clients gated out       : {p['clients_gated_out']}  (per-client AUC < 0.60)")
print(f"\nUncapped top 50 : precision {p['top50_uncapped']['precision']:.3f} · "
      f"largest client {p['top50_uncapped']['largest_client_share']:.0%} · "
      f"{p['top50_uncapped']['distinct_clients']} clients")
print(f"Capped   top 50 : precision {p['top50_capped']['precision']:.3f} · "
      f"largest client {p['top50_capped']['largest_client_share']:.0%} · "
      f"{p['top50_capped']['distinct_clients']} clients")
print(f"\nsealed base rate: {p['sealed_base_rate']}")

Where the queue abstains rather than guesses:
  clients measured        : 28
  clients gated out       : 6  (per-client AUC < 0.60)

Uncapped top 50 : precision 0.900 · largest client 40% · 6 clients
Capped   top 50 : precision 0.740 · largest client 10% · 13 clients

sealed base rate: 0.5622


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

**What a FlyRank editor would do with this tomorrow.**

Work the capped queue top-down. Each row states an action chosen by *which evidence fired*, not
by score height:

| Action | Evidence | What to do | Pages |
|---|---|---|---|
| **`review_now`** | already sliding **and** on page 1–2 | open it, find what changed, refresh if warranted | 53,978 |
| **`investigate_volatility`** | position unstable, no clear direction | look for technical or SERP-layout causes *before* touching content | 20,622 |
| **`defend_position`** | visible, shallow, currently stable | leave the content alone; watch weekly | 14,222 |
| **`monitor`** | everything else | nothing this cycle | 27,834 |

**Three rules that come with the queue.**

1. **Cap each client at 5 pages per 50.** Uncapped, one client takes 38% of the top 50 and only
   6 portfolios appear. The cap costs **10 points of precision (0.860 → 0.760)** and buys a queue
   **12 portfolio owners** can each act on. That trade is a judgement call, and it is stated as
   one.
2. **Abstain where skill was never measured.** Six of 28 measured clients fall below AUC 0.60 and
   should be served no queue at all. A queue that is wrong for a portfolio is worse than no queue.
3. **Check before acting.** Did the page change, or did the SERP? Is the decline in queries you
   care about? Is there a technical cause? The reason codes are the model's argument — a reviewer
   should be able to disagree with a specific claim rather than with "the model".

**Never automate off this score:** no auto-rewrite, no auto-deindex or delete, no client-facing
performance reporting or billing, no probability language, no causal claim.

**Confidence.** Directional and modest. The ranking beat both a base rate and a hand-written rule
on a sealed future month, twice, and the effect is small enough that a transparent rule remains a
legitimate fallback — which is exactly what the monitoring plan reverts to when a trigger fires.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [8]:
DOCS_IMG = DOCS/"img"; DOCS_IMG.mkdir(parents=True, exist_ok=True)
figs = ["precision_at_k_sealed.svg", "action_mix.svg", "client_concentration.svg"]
for f in figs:
    shutil.copyfile(FIG/f, DOCS_IMG/f)
    print("paper embeds:", rel(DOCS_IMG/f))

summary = pd.DataFrame({
    "metric": ["base rate (dev)", "baseline rule AUC", "model AUC (grouped)",
               "baseline rule P@50", "model P@50 (grouped)",
               "model AUC (sealed Jun)", "model P@50 (sealed Jun)",
               "model AUC (random split, NOT the claim)"],
    "value": [R["w05_model_metrics"]["base_rate"],
              R["w05_model_metrics"]["baseline_rule"]["roc_auc"],
              R["w05_model_metrics"]["grouped"]["random_forest"]["roc_auc"],
              R["w05_model_metrics"]["baseline_rule"]["P@50"],
              R["w05_model_metrics"]["grouped"]["random_forest"]["P@50"],
              R["w06_sealed_test_metrics"]["results"]["SEALED Jun-2026, all clients"]["roc_auc"],
              R["w06_sealed_test_metrics"]["results"]["SEALED Jun-2026, all clients"]["P@50"],
              R["w05_model_metrics"]["random_split_diagnostic"]["random_forest"]]})
summary.to_csv(OUT/"capstone_headline_numbers.csv", index=False)
print("\n" + summary.to_string(index=False))
print(f"\nwrote {rel(OUT/'capstone_headline_numbers.csv')}")

paper embeds: docs/img/precision_at_k_sealed.svg
paper embeds: docs/img/action_mix.svg
paper embeds: docs/img/client_concentration.svg

                                 metric  value
                        base rate (dev) 0.5673
                      baseline rule AUC 0.6366
                    model AUC (grouped) 0.6507
                     baseline rule P@50 0.8200
                   model P@50 (grouped) 0.8800
                 model AUC (sealed Jun) 0.6923
                model P@50 (sealed Jun) 0.9000
model AUC (random split, NOT the claim) 0.7766

wrote work/outputs/capstone_headline_numbers.csv


---

# ML-12 — Demo, social cut, and employer summary

## A. Five-minute demo outline

| Min | Beat | What is on screen | The line that lands |
|---|---|---|---|
| **0:00–0:45** | **The problem** | a portfolio dashboard sorted by traffic | "A strategist can open 30 pages this week. Sorting by traffic shows the *biggest* pages, not the ones *slipping*. Which 30?" |
| **0:45–1:30** | **The setup** | the ML-04 contract table: feature window, outcome window, sealed month | "79 million daily rows. One row per page. Features from Jan–March, label from April — not one day of overlap. June sealed and untouched until the end." |
| **1:30–2:15** | **The honest baseline** | the five-condition rule and its P@50 | "Before any model: a rule you can read. Already sliding, unstable, shallow, visible. It gets 76% of its top 50 right against a 57% base rate." |
| **2:15–3:15** | **The twist — the split** | grouped vs random table, side by side | "My random forest hit 0.96 in its top 50 — until I stopped letting it see the same client on both sides of the split. Then it fell to 0.88. Same model, same rows. A third of the score was memorising clients." |
| **3:15–4:00** | **What survived** | sealed June results + the frozen rule beside it | "On a sealed future month: 0.900. The hand-written rule: 0.780. But go 500 pages deep and the rule *wins* — 0.916 to 0.844. They're better at different depths, and that's the honest story." |
| **4:00–4:40** | **The product** | the capped queue, reason codes visible | "Cap each client at 5 per 50. Costs 16 points of precision, turns one client's problem into a queue 13 people can work. And it abstains for 6 clients where it has no measured skill." |
| **4:40–5:00** | **The close** | the limitations slide | "It ranks, it doesn't predict. It's decision-support for a review queue — and I can tell you exactly where it stops working." |

**If one question comes:** *"Why is the model barely better than the rule?"* — "Because most of
the signal is one thing: whether the page was already sliding. A rule captures that. The forest
adds the interactions around it. Pretending that gap is bigger would be the dishonest version of
this talk."

---

## B. Social post cut

> Everyone refreshes the oldest content first. On 79M rows of real search data, that is **worse than picking at random**.
>
> Ordering a refresh queue by "days since last update" got **0.46** precision at the top — against
> a 0.56 base rate. Ordering the same pages by whether they were *already losing position* got
> **0.90**.
>
> The original split let the model see *other pages from the same client* during training. So it
> wasn't learning what precedes a decline; it was learning to recognise the client and recite
> that client's average outcome. Grouping the split by client cut apparent skill above base rate
> roughly in half.
>
> What survived is smaller and real: on a **sealed** future month (79M daily rows, 116k pages),
> ranking by observed position-momentum put **90% declining pages in the top 50** against a 56%
> base rate. A four-condition rule you could write on a napkin got **78%** — and beat the model
> outright deeper in the queue. The honest headline isn't "ML wins", it's "they win at different
> depths".
>
> The most useful findings were the two that said *no*.
>
> 📊 [chart: precision@K, sealed month] · full write-up + notebooks in the repo.

*(Chart to attach: `docs/img/precision_at_k_sealed.svg`.)*

---

## C. Employer-facing summary — three sentences

> I built a search-ranking risk model on a **79-million-row** pseudonymized web-performance
> warehouse, querying it in place with DuckDB over remote Parquet rather than downloading it, and
> shipped it as a reason-coded review queue with an abstention gate and drift monitors.
>
> Working out-of-fold on **clients the model had never seen** and validating on a **sealed future
> month**, it surfaced declining pages at **1.6× the base rate** (precision@50 of 0.90 against a
> 0.56 base) — while a transparent hand-written rule reached 0.78 at the top and beat the model
> outright from 500 pages deep, which I report because it is the honest comparison.
>
> The results I am most pleased with are two negatives: I showed the industry-standard "refresh
> the oldest content" heuristic performs worse than random on this data, and caught a field that
> looked like page staleness but was actually future information present in 82% of rows.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
